# Document Processing for RAG — Academic Rules PDF

**Pipeline:** PDF → Extract → Clean → Preserve Structure → Chunk → Add Metadata → Validate

This notebook converts a PDF of academic rules (`Aturan_Akademik.pdf`) into a clean, chunked, metadata-rich CSV ready for embedding in a RAG pipeline.

## What You'll Learn
- Extracting text from PDFs with `pypdf`
- Cleaning PDF noise without destroying document structure
- Comparing **fixed-size** vs **structure-aware** chunking
- Building a parser that attaches BAB/Pasal metadata to each chunk
- Validating chunk quality before embedding

## Requirements
- Python 3.10+
- `pypdf`, `pandas`

## Author
- Huzaifa Akmal ([@huzaifaakmal](https://github.com/huzaifaakmal))

## Table of Contents
1. [Setup](#1-setup)
2. [Extract Text from PDF](#2-extract-text-from-pdf)
3. [Clean Text](#3-clean-text)
4. [Fixed-Size Chunking](#4-fixed-size-chunking)
5. [Structure-Aware Chunking](#5-structure-aware-chunking)
6. [Parse Metadata](#6-parse-metadata)
7. [Build Context Text](#7-build-context-text)
8. [Validate Chunks](#8-validate-chunks)
9. [Experiment Comparison](#9-experiment-comparison)
10. [Conclusion](#10-conclusion)

In [1]:
%pip install pypdf pandas --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\Users\ASUS\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [7]:
import re
import pandas as pd
from pathlib import Path
from pypdf import PdfReader

## 2. Extract Text from PDF

Read the source PDF page by page using `pypdf`. If the PDF is text-based (not a scan), `extract_text()` returns readable text.

The raw extraction is saved to `output/raw_text.txt` for inspection.

In [8]:
# Paths relative to the notebook/ folder
PDF_PATH = Path("../data/Aturan_Akademik.pdf")
OUTPUT_DIR = Path("../output")
OUTPUT_DIR.mkdir(exist_ok=True)

if not PDF_PATH.exists():
    raise FileNotFoundError(
        f"{PDF_PATH} not found. Make sure data/Aturan_Akademik.pdf exists."
    )

reader = PdfReader(str(PDF_PATH))

pages = []
for page_no, page in enumerate(reader.pages, start=1):
    text = page.extract_text() or ""
    pages.append({"page": page_no, "text": text})

print("Total pages:", len(pages))
print("\n--- First page preview ---")
print(pages[0]["text"][:1000])

# Save raw extracted text
raw_text = "\n".join(p["text"] for p in pages)
(OUTPUT_DIR / "raw_text.txt").write_text(raw_text, encoding="utf-8")
print(f"\nSaved {OUTPUT_DIR / 'raw_text.txt'} ({len(raw_text)} chars)")

Total pages: 10

--- First page preview ---
Aturan Akademik Sintetis — Dataset Praktikum RAG 
PERATURAN AKADEMIK MAHASISWA 
Dokumen Sintetis Terstruktur untuk Praktikum Document Processing dan RAG 
Versi Bahasa Formal — Tahun Akademik 2026/2027 
 
Catatan: Dokumen ini disusun sebagai dataset pembelajaran dan bukan merupakan peraturan resmi 
institusi tertentu. 
  

Saved ..\output\raw_text.txt (20229 chars)


In [9]:
def clean_text(text):
    # Normalize line endings
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Collapse horizontal whitespace (spaces/tabs) but keep newlines
    text = re.sub(r"[ \t]+", " ", text)

    # Remove trailing spaces after newlines
    text = re.sub(r"\n *", "\n", text)

    # Collapse 3+ newlines into 2
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Remove standalone page numbers
    text = re.sub(r"(?m)^\s*\d+\s*$", "", text)

    return text.strip()


cleaned_text = clean_text(raw_text)
(OUTPUT_DIR / "clean_text.txt").write_text(cleaned_text, encoding="utf-8")

print("--- Cleaned preview ---")
print(cleaned_text[:1500])
print(f"\nTotal characters after cleaning: {len(cleaned_text)}")

--- Cleaned preview ---
Aturan Akademik Sintetis — Dataset Praktikum RAG 
PERATURAN AKADEMIK MAHASISWA 
Dokumen Sintetis Terstruktur untuk Praktikum Document Processing dan RAG 
Versi Bahasa Formal — Tahun Akademik 2026/2027 

Catatan: Dokumen ini disusun sebagai dataset pembelajaran dan bukan merupakan peraturan resmi 
institusi tertentu. 

Aturan Akademik Sintetis — Dataset Praktikum RAG 
BAB I — KETENTUAN UMUM 
Pasal 1 — Definisi 
(1) Satuan Kredit Semester, yang selanjutnya disingkat SKS, merupakan ukuran beban belajar mahasiswa 
dalam satu semester. 
(2) Indeks Prestasi Semester, yang selanjutnya disingkat IPS, merupakan ukuran capaian akademik 
mahasiswa yang dihitung berdasarkan seluruh mata kuliah yang ditempuh dalam satu semester. 
(3) Indeks Prestasi Kumulatif, yang selanjutnya disingkat IPK, merupakan ukuran capaian akademik 
mahasiswa yang dihitung berdasarkan seluruh mata kuliah yang telah ditempuh. 
(4) Nilai D merupakan nilai lulus terbatas, sedangkan nilai E merupakan n

In [10]:
def fixed_chunks(text, chunk_size=500, overlap=100):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks


chunks_A = fixed_chunks(cleaned_text, chunk_size=300, overlap=50)
chunks_B = fixed_chunks(cleaned_text, chunk_size=800, overlap=150)

print(f"Strategy A (300/50) -> {len(chunks_A)} chunks")
print(f"Strategy B (800/150) -> {len(chunks_B)} chunks")

print("\n--- First chunk of A ---")
print(chunks_A[0])
print("\n--- First chunk of B ---")
print(chunks_B[0])

Strategy A (300/50) -> 81 chunks
Strategy B (800/150) -> 32 chunks

--- First chunk of A ---
Aturan Akademik Sintetis — Dataset Praktikum RAG 
PERATURAN AKADEMIK MAHASISWA 
Dokumen Sintetis Terstruktur untuk Praktikum Document Processing dan RAG 
Versi Bahasa Formal — Tahun Akademik 2026/2027 

Catatan: Dokumen ini disusun sebagai dataset pembelajaran dan bukan merupakan peraturan resmi 
in

--- First chunk of B ---
Aturan Akademik Sintetis — Dataset Praktikum RAG 
PERATURAN AKADEMIK MAHASISWA 
Dokumen Sintetis Terstruktur untuk Praktikum Document Processing dan RAG 
Versi Bahasa Formal — Tahun Akademik 2026/2027 

Catatan: Dokumen ini disusun sebagai dataset pembelajaran dan bukan merupakan peraturan resmi 
institusi tertentu. 

Aturan Akademik Sintetis — Dataset Praktikum RAG 
BAB I — KETENTUAN UMUM 
Pasal 1 — Definisi 
(1) Satuan Kredit Semester, yang selanjutnya disingkat SKS, merupakan ukuran beban belajar mahasiswa 
dalam satu semester. 
(2) Indeks Prestasi Semester, yang selanju

## 5. Structure-Aware Chunking

For academic rules, the boundary between Pasal is a **strong semantic signal**. Instead of splitting by character count, split by structure.

We use `re.split` with a lookahead pattern `(?=Pasal\s+\d+)` — this splits before every "Pasal N" without consuming the marker itself.

**Weakness:** a BAB heading that appears before the first Pasal may attach to the previous chunk. This is fixed in Section 6 when we parse metadata explicitly.

In [11]:
pattern = r"(?=Pasal\s+\d+)"
parts = re.split(pattern, cleaned_text)

# Keep only chunks that begin with "Pasal N — Title" (real headings)
article_chunks = [
    p.strip() for p in parts
    if re.match(r"^Pasal\s+\d+\s*[—\-–]", p.strip())
]

print(f"Structure-aware -> {len(article_chunks)} chunks\n")

for i, chunk in enumerate(article_chunks[:3]):
    print(f"--- Article Chunk {i} ---")
    print(chunk[:400])
    print()

Structure-aware -> 41 chunks

--- Article Chunk 0 ---
Pasal 1 — Definisi 
(1) Satuan Kredit Semester, yang selanjutnya disingkat SKS, merupakan ukuran beban belajar mahasiswa 
dalam satu semester. 
(2) Indeks Prestasi Semester, yang selanjutnya disingkat IPS, merupakan ukuran capaian akademik 
mahasiswa yang dihitung berdasarkan seluruh mata kuliah yang ditempuh dalam satu semester. 
(3) Indeks Prestasi Kumulatif, yang selanjutnya disingkat IPK, meru

--- Article Chunk 1 ---
Pasal 2 — Beban dan Masa Studi 
(1) Program sarjana memiliki beban studi sekurang-kurangnya 144 SKS. 
(2) Masa studi normal program sarjana adalah 8 semester. 
(3) Masa studi mahasiswa tidak boleh melebihi 14 semester aktif. Semester cuti akademik yang telah 
memperoleh persetujuan tidak diperhitungkan sebagai semester aktif. 
(4) Mahasiswa yang telah melampaui 14 semester aktif dinyatakan tidak m

--- Article Chunk 2 ---
Pasal 3 — Registrasi Administrasi 
(1) Setiap mahasiswa wajib menyelesaikan registrasi adminis

## 6. Parse Metadata (BAB, Pasal, chunk_id)

We walk through the cleaned text line by line, tracking the current BAB and Pasal. Each flush creates a row with:
- `bab` — the chapter the Pasal belongs to
- `pasal` — the article heading
- `text` — the full rule content

Then we assign a `chunk_id` (AKD-001 … AKD-042), a `source` label, and a `char_count`.

In [12]:
rows = []
current_bab = None
current_pasal = None
buffer = []

def flush():
    if current_pasal and buffer:
        rows.append({
            "bab": current_bab,
            "pasal": current_pasal,
            "text": " ".join(buffer).strip()
        })

for line in cleaned_text.splitlines():
    line = line.strip()
    if not line:
        continue

    if re.match(r"^BAB\s+[IVXLCDM]+\b", line, flags=re.I):
        flush()
        buffer = []
        current_bab = line
        current_pasal = None
    elif re.match(r"^Pasal\s+\d+\s*[—\-–]", line, flags=re.I):
        flush()
        buffer = []
        current_pasal = line
    else:
        buffer.append(line)

flush()

df = pd.DataFrame(rows)
df["chunk_id"] = [f"AKD-{i:03d}" for i in range(1, len(df) + 1)]
df["source"] = "aturan_akademik"
df["char_count"] = df["text"].str.len()

print(f"Total articles parsed: {len(df)}")
df.head()

Total articles parsed: 41


,bab,pasal,text,chunk_id,source,char_count
0,BAB I — KETENTUAN UMUM,Pasal 1 — Definisi,"(1) Satuan Kredit Semester, yang selanjutnya d...",AKD-001,aturan_akademik,573
1,BAB I — KETENTUAN UMUM,Pasal 2 — Beban dan Masa Studi,(1) Program sarjana memiliki beban studi sekur...,AKD-002,aturan_akademik,470
2,BAB II — REGISTRASI DAN KARTU RENCANA STUDI,Pasal 3 — Registrasi Administrasi,(1) Setiap mahasiswa wajib menyelesaikan regis...,AKD-003,aturan_akademik,348
3,BAB II — REGISTRASI DAN KARTU RENCANA STUDI,Pasal 4 — Pengisian Kartu Rencana Studi,(1) Kartu Rencana Studi memuat mata kuliah yan...,AKD-004,aturan_akademik,491
4,BAB II — REGISTRASI DAN KARTU RENCANA STUDI,Pasal 5 — Perubahan Kartu Rencana Studi,(1) Mahasiswa dapat menambah atau membatalkan ...,AKD-005,aturan_akademik,408


## 7. Build `context_text`

The final dataset needs one more column: `context_text`. This merges BAB + Pasal + text into a single string.

**Why?** When the chunk is embedded in Practicum 3, the embedding should capture *which rule under which chapter* — not just the rule content.

The result is saved to `output/chunks_aturan_akademik.csv`.

In [6]:
df["context_text"] = (
    df["bab"].fillna("") + "\n" +
    df["pasal"].fillna("") + "\n" +
    df["text"].fillna("")
)

df.to_csv(OUTPUT_DIR / "chunks_aturan_akademik.csv", index=False, encoding="utf-8-sig")

print(f"Saved {OUTPUT_DIR / 'chunks_aturan_akademik.csv'}")
df[["chunk_id", "bab", "pasal", "char_count"]]

NameError: name 'df' is not defined

## 8. Validate Chunk Quality

Simple rules flag problematic chunks before they reach the embedding stage:

| Issue | Threshold |
|-------|-----------|
| `too_short` | < 30 chars |
| `too_long` | > 1500 chars |
| `missing_pasal` | empty `pasal` field |
| `missing_bab` | empty `bab` field |

A chunk flagged `OK` is ready for the next practicum.

In [10]:
def validate_chunk(row):
    issues = []
    if len(row["text"]) < 30:
        issues.append("too_short")
    if len(row["text"]) > 1500:
        issues.append("too_long")
    if not row["pasal"]:
        issues.append("missing_pasal")
    if not row["bab"]:
        issues.append("missing_bab")
    return ", ".join(issues) if issues else "OK"


df["validation"] = df.apply(validate_chunk, axis=1)
print(df[["chunk_id", "char_count", "validation"]])

   chunk_id  char_count validation
0   AKD-001         573         OK
1   AKD-002         470         OK
2   AKD-003         348         OK
3   AKD-004         491         OK
4   AKD-005         408         OK
5   AKD-006         432         OK
6   AKD-007         190         OK
7   AKD-008         497         OK
8   AKD-009         394         OK
9   AKD-010         345         OK
10  AKD-011         497         OK
11  AKD-012         203         OK
12  AKD-013         233         OK
13  AKD-014         275         OK
14  AKD-015         566         OK
15  AKD-016         240         OK
16  AKD-017         238         OK
17  AKD-018         254         OK
18  AKD-019         475         OK
19  AKD-020         209         OK
20  AKD-021         280         OK
21  AKD-022         403         OK
22  AKD-023         209         OK
23  AKD-024         198         OK
24  AKD-025         783         OK
25  AKD-026         336         OK
26  AKD-027         273         OK
27  AKD-028         

In [11]:
chunks_C = df  # use parsed DataFrame so count matches chunks_aturan_akademik.csv

experiments = [
    {
        "Strategy": "A",
        "Configuration": "Fixed-size 300 chars, overlap 50",
        "Chunk Count": len(chunks_A),
        "Rules Cut?": "Yes",
        "Clear Metadata?": "No",
        "Notes": "Narrow context, many rules cut mid-sentence",
    },
    {
        "Strategy": "B",
        "Configuration": "Fixed-size 800 chars, overlap 150",
        "Chunk Count": len(chunks_B),
        "Rules Cut?": "Sometimes",
        "Clear Metadata?": "No",
        "Notes": "Wider context, but some articles may be merged",
    },
    {
        "Strategy": "C",
        "Configuration": "Structure-aware (1 Pasal = 1 chunk)",
        "Chunk Count": len(chunks_C),
        "Rules Cut?": "No",
        "Clear Metadata?": "Yes (BAB, Pasal)",
        "Notes": "Each rule stays intact and traceable",
    },
]

df_experiments = pd.DataFrame(experiments)
print(df_experiments.to_string(index=False))

Strategy                       Configuration  Chunk Count Rules Cut?  Clear Metadata?                                          Notes
       A    Fixed-size 300 chars, overlap 50           81        Yes               No    Narrow context, many rules cut mid-sentence
       B   Fixed-size 800 chars, overlap 150           32  Sometimes               No Wider context, but some articles may be merged
       C Structure-aware (1 Pasal = 1 chunk)           41         No Yes (BAB, Pasal)           Each rule stays intact and traceable


## Best vs Problematic Chunks

### Best chunks (structure-aware)
- **AKD-005** — `BAB III — BEBAN STUDI | Pasal 5`: complete IPS → SKS mapping in one unit
- **AKD-008** — `BAB III — BEBAN STUDI | Pasal 8`: self-contained rule on academic evaluation
- **AKD-041** — `BAB XI — INTEGRITAS AKADEMIK | Pasal 40`: complete rule on sanction impact

### Problematic chunk
- **AKD-042** — `BAB XII — KETENTUAN PENUTUP | Pasal 41` — **2027 chars** — flagged `too_long`
  - Cause: Pasal 41 has three long ayat and is the final Pasal (no following article to trigger a flush)
  - Impact: ~500 tokens may exceed embedding model limits
  - Fix: split by ayat markers `(1)`, `(2)`, `(3)` when a Pasal exceeds 1500 chars

## 10. Conclusion

**The most suitable strategy is Structure-aware chunking (1 Pasal = 1 chunk).**

For a document as highly structured as academic rules, character-based chunking is inadequate — it cuts rules mid-sentence and separates conditions from consequences. Structure-aware chunking preserves the natural semantic unit (the Pasal) and attaches explicit BAB metadata, producing **41 chunks where 40 are fully valid**.

This makes the resulting CSV directly usable for embedding and retrieval in **Practicum 3 — Embedding & Vector Search**.

In [12]:
print("=== OUTPUT FILES ===")
for f in sorted(Path("output").iterdir()):
    print(f"OK  {f.name}  ({f.stat().st_size} bytes)")

print("\n=== FINAL DATASET ===")
print(f"Total chunks   : {len(df)}")
print(f"Columns        : {list(df.columns)}")
print(f"Avg chunk size : {df['char_count'].mean():.0f} chars")
print(f"Median size    : {df['char_count'].median():.0f} chars")
print(f"Max size       : {df['char_count'].max()} chars")
print(f"Valid chunks   : {(df['validation'] == 'OK').sum()}/{len(df)}")
print("\nReady for Practicum 3 — Embedding & Vector Search.")

=== OUTPUT FILES ===
OK  chunks_aturan_akademik.csv  (42873 bytes)
OK  clean_text.txt  (20664 bytes)
OK  raw_text.txt  (20670 bytes)

=== FINAL DATASET ===
Total chunks   : 41
Columns        : ['bab', 'pasal', 'text', 'chunk_id', 'source', 'char_count', 'context_text', 'validation']
Avg chunk size : 432 chars
Median size    : 384 chars
Max size       : 2027 chars
Valid chunks   : 40/41

Ready for Practicum 3 — Embedding & Vector Search.
